# Task 3: Most Effective Ensemble Methods

## 1. Setup and Installations

In [1]:
!pip install ranx

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.0/859.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.0/135.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.2 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=21f0009ebc9ae115cbe11bf5f822e389576e604995eee0b84d47e14758c3f209
  Stored in directory: /root/.cache/pip/wheels/63/f9/dc/2dd16d3330e327236e4d407941975c42d5159d200cdb7922d8
  Created wheel for cbor: filename=cbor-1.0.0-cp311-cp311-linux_x86_64.whl size=5393

In [2]:
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 29.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [4]:
import ranx
import os
from google.colab import drive
import pandas as pd
import logging # For data download script
from sentence_transformers import util # For util.http_get
from collections import defaultdict # For data download script
import gzip # For data download script

# Setup logging for the data download script
logger = logging.getLogger(__name__) # Use __name__ for module-level logger
logger.setLevel(logging.INFO) # Set to INFO or DEBUG as needed
if not logger.hasHandlers(): # Avoid adding multiple handlers if re-running cell
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter('%(asctime)s - %(message)s',datefmt='%Y-%m-%d %H:%M:%S'))
    logger.addHandler(handler)

In [6]:
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Define Paths and Download/Load Data

In [7]:
# Base path to your project directory on Google Drive for model run files
base_drive_path = "/content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/"

# Paths to the model run files (generated in Task 1/evaluating.ipynb)
minilm_model_dir = os.path.join(base_drive_path, "finetuned_models/cross-encoder-ms-marco-MiniLM-L-2-v2")
distilroberta_model_dir = os.path.join(base_drive_path, "finetuned_models/cross-encoder-distilroberta-base")
tinybert_model_dir = os.path.join(base_drive_path, "finetuned_models/cross-encoder-ms-marco-tinybert-l-2-v2")

# Construct full paths to the .run files
# It's assumed the run files are named 'ranking.run' within their respective model directories
# If your evaluating.ipynb saved them with date-time stamps in the filename, adjust these paths accordingly.
minilm_run_path = ""
distilroberta_run_path = ""
tinybert_run_path = ""

# Find the most recent .run file in each directory if they have timestamps
def find_latest_run_file(model_dir):
    run_files = [f for f in os.listdir(model_dir) if f.endswith("ranking.run")]
    if not run_files:
        print(f"WARNING: No 'ranking.run' file found in {model_dir}")
        return None
    # Sort by modification time (most recent first) if multiple, or just pick one if names are static
    if len(run_files) > 1:
        # This assumes filenames without timestamps, or you sort them by timestamp if present
        # For now, just picking the first one if names are static like 'ranking.run'
        # If names are dynamic, a more robust sorting by date in filename is needed
        print(f"Multiple .run files in {model_dir}, using {run_files[0]}. Adjust if this is not the correct one.")
        return os.path.join(model_dir, run_files[0])
    return os.path.join(model_dir, run_files[0])

if os.path.exists(minilm_model_dir):
    minilm_run_path = find_latest_run_file(minilm_model_dir)
else:
    print(f"Directory not found: {minilm_model_dir}")

if os.path.exists(distilroberta_model_dir):
    distilroberta_run_path = find_latest_run_file(distilroberta_model_dir)
else:
    print(f"Directory not found: {distilroberta_model_dir}")

if os.path.exists(tinybert_model_dir):
    tinybert_run_path = find_latest_run_file(tinybert_model_dir)
else:
    print(f"Directory not found: {tinybert_model_dir}")


print(f"Using MiniLM Run Path: {minilm_run_path}")
print(f"Using DistilRoberta Run Path: {distilroberta_run_path}")
print(f"Using TinyBERT Run Path: {tinybert_run_path}")


Using MiniLM Run Path: /content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-ms-marco-MiniLM-L-2-v2/ranking.run
Using DistilRoberta Run Path: /content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-distilroberta-base/ranking.run
Using TinyBERT Run Path: /content/drive/MyDrive/cross-encoder-reranker-ir-course-2023/finetuned_models/cross-encoder-ms-marco-tinybert-l-2-v2/ranking.run


### Download and Prepare TREC DL 2019 Data (Qrels, Queries, Candidate Passages)

In [8]:
data_folder = 'trec2019-data'
os.makedirs(data_folder, exist_ok=True)

logger.info("--- Downloading TREC DL 2019 Data ---")

#Read test queries
queries = {}
queries_filepath = os.path.join(data_folder, 'msmarco-test2019-queries.tsv.gz')
if not os.path.exists(queries_filepath):
    logger.info("Download "+os.path.basename(queries_filepath))
    util.http_get('https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-test2019-queries.tsv.gz', queries_filepath)
else:
    logger.info(f"{os.path.basename(queries_filepath)} already exists.")

with gzip.open(queries_filepath, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        qid, query = line.strip().split("\t")
        queries[qid] = query

#Read which passages are relevant
relevant_docs_dict_for_pytrec = defaultdict(lambda: defaultdict(int)) # For pytrec_eval format
qrels_filepath = os.path.join(data_folder, '2019qrels-pass.txt')

if not os.path.exists(qrels_filepath):
    logger.info("Download "+os.path.basename(qrels_filepath))
    util.http_get('https://trec.nist.gov/data/deep/2019qrels-pass.txt', qrels_filepath)
else:
    logger.info(f"{os.path.basename(qrels_filepath)} already exists.")

with open(qrels_filepath) as fIn:
    for line in fIn:
        qid, _, pid, score = line.strip().split()
        score = int(score)
        if score > 0: # Consider only relevant documents for TREC DL
            relevant_docs_dict_for_pytrec[qid][pid] = score

# Read the top 1000 passages that are supposed to be re-ranked (not strictly needed for ranx.evaluate if runs are complete, but good to have context)
passage_filepath = os.path.join(data_folder, 'msmarco-passagetest2019-top1000.tsv.gz')
if not os.path.exists(passage_filepath):
    logger.info("Download "+os.path.basename(passage_filepath))
    util.http_get('https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-passagetest2019-top1000.tsv.gz', passage_filepath)
else:
    logger.info(f"{os.path.basename(passage_filepath)} already exists.")

logger.info(f"Loaded {len(queries)} test queries.")
logger.info(f"Loaded relevance judgments for {len(relevant_docs_dict_for_pytrec)} queries.")
logger.info("--- TREC DL 2019 Data Preparation Complete ---")

INFO:__main__:--- Downloading TREC DL 2019 Data ---
INFO:__main__:Download msmarco-test2019-queries.tsv.gz


  0%|          | 0.00/4.28k [00:00<?, ?B/s]

INFO:__main__:Download 2019qrels-pass.txt


0.00B [00:00, ?B/s]

INFO:__main__:Download msmarco-passagetest2019-top1000.tsv.gz


  0%|          | 0.00/26.6M [00:00<?, ?B/s]

INFO:__main__:Loaded 200 test queries.
INFO:__main__:Loaded relevance judgments for 43 queries.
INFO:__main__:--- TREC DL 2019 Data Preparation Complete ---


### Load Runs and Qrels for Ranx

In [9]:
# Load Qrels for ranx
qrels = ranx.Qrels.from_file(qrels_filepath, kind="trec")

# Load individual model runs if paths were found
runs_to_fuse = []
run_names = []

if minilm_run_path and os.path.exists(minilm_run_path):
    run_minilm = ranx.Run.from_file(minilm_run_path, kind="trec")
    runs_to_fuse.append(run_minilm)
    run_names.append("MiniLM")
else:
    print(f"Skipping MiniLM run as file was not found: {minilm_run_path}")

if distilroberta_run_path and os.path.exists(distilroberta_run_path):
    run_distilroberta = ranx.Run.from_file(distilroberta_run_path, kind="trec")
    runs_to_fuse.append(run_distilroberta)
    run_names.append("DistilRoberta")
else:
    print(f"Skipping DistilRoberta run as file was not found: {distilroberta_run_path}")

if tinybert_run_path and os.path.exists(tinybert_run_path):
    run_tinybert = ranx.Run.from_file(tinybert_run_path, kind="trec")
    runs_to_fuse.append(run_tinybert)
    run_names.append("TinyBERT")
else:
    print(f"Skipping TinyBERT run as file was not found: {tinybert_run_path}")

if len(runs_to_fuse) < 2:
  print("\nERROR: Less than two run files were found. Ensemble methods require at least two runs. Please check the paths and ensure the run files exist.")
else:
  print(f"\nSuccessfully loaded {len(runs_to_fuse)} runs for fusion: {run_names}")


Successfully loaded 3 runs for fusion: ['MiniLM', 'DistilRoberta', 'TinyBERT']


## 3. Select and Apply Ensemble Methods

In [15]:
if len(runs_to_fuse) >= 2: # Proceed only if we have enough runs
    # Define the 5 selected ensemble methods as strings (method names for ranx.fuse)
    # These aliases are taken from the ranx documentation: https://amenra.github.io/ranx/fusion/
    ensemble_method_names = {
        "CombSUM": "sum",
        "CombMNZ": "mnz",
        "RRF": "rrf",        # Reciprocal Rank Fusion
        "BordaFuse": "bordafuse",
        "LogISRFuse": "log_isr" # Log Information Secure Ratio
    }

    # Check if LogISRFuse is available, otherwise fallback or skip
    # Note: ranx documentation lists Log_ISR with an underscore.
    # We will try 'log_isr' first, then 'isr' as a fallback if 'log_isr' causes issues due to naming or availability.
    # Directly checking for method availability within ranx.fuse's capabilities is tricky without trying to run it.
    # We'll rely on the string alias and handle potential errors if a method isn't recognized.

    metrics = ["ndcg@10", "recall@100", "map@1000"]
    results_summary = []

    # Define a default normalization strategy (as shown in ranx.fuse examples)
    # Min-Max normalization is a common choice.
    normalization_strategy = "min-max"

    print("\n--- Individual Model Performance ---")
    for i, run in enumerate(runs_to_fuse):
        run.name = run_names[i] # Assign name for ranx report
        individual_scores = ranx.evaluate(qrels, run, metrics)
        print(f"Model: {run.name}")
        for metric, score in individual_scores.items():
            print(f"  {metric}: {score:.4f}")
        print("---")
        results_summary.append({
            "Method": run.name,
            "ndcg@10": individual_scores.get("ndcg@10"),
            "recall@100": individual_scores.get("recall@100"),
            "map@1000": individual_scores.get("map@1000")
        })


    print("\n--- Ensemble Method Performance ---")
    for display_name, method_alias in ensemble_method_names.items():
        print(f"Fusing with: {display_name} (ranx method: '{method_alias}')")

        try:
            fusion_params = {}
            if method_alias == "rrf": # RRF takes a 'k' parameter
                fusion_params['k'] = 60 # A common default for RRF

            # Apply fusion
            fused_run = ranx.fuse(
                runs=runs_to_fuse,
                norm=normalization_strategy, # Apply normalization before fusion
                method=method_alias,
                params=fusion_params if fusion_params else None # Pass params only if they exist
            )

            fused_run.name = display_name # Use the descriptive name for reports

            # Evaluate the fused run
            scores = ranx.evaluate(qrels, fused_run, metrics)

            print(f"Results for {display_name}:")
            for metric, score in scores.items():
                print(f"  {metric}: {score:.4f}")
            results_summary.append({
                "Method": display_name,
                "ndcg@10": scores.get("ndcg@10"),
                "recall@100": scores.get("recall@100"),
                "map@1000": scores.get("map@1000")
            })
        except Exception as e:
            print(f"Error applying {display_name} (method: {method_alias}): {e}")
            # As a fallback for LogISRFuse if 'log_isr' isn't recognized and 'isr' is
            if method_alias == "log_isr" and 'isr' in dir(ranx.fuse): # Check if isr exists as a direct function
                print(f"Attempting fallback to 'isr' for {display_name}")
                try:
                    fused_run_fallback = ranx.fuse(
                        runs=runs_to_fuse,
                        norm=normalization_strategy,
                        method="isr"
                    )
                    fused_run_fallback.name = f"{display_name} (ISR Fallback)"
                    scores_fallback = ranx.evaluate(qrels, fused_run_fallback, metrics)
                    print(f"Results for {display_name} (ISR Fallback):")
                    for metric, score in scores_fallback.items():
                        print(f"  {metric}: {score:.4f}")
                    results_summary.append({
                        "Method": f"{display_name} (ISR Fallback)",
                        "ndcg@10": scores_fallback.get("ndcg@10"),
                        "recall@100": scores_fallback.get("recall@100"),
                        "map@1000": scores_fallback.get("map@1000")
                    })
                except Exception as e_fallback:
                    print(f"Error applying fallback 'isr' for {display_name}: {e_fallback}")
        print("---")
else:
    print("Skipping ensemble methods due to insufficient loaded runs.")


--- Individual Model Performance ---


/usr/local/lib/python3.11/dist-packages/ranx/metrics/ndcg.py:72: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _ndcg(qrels[i], run[i], k, rel_lvl, jarvelin)


Model: MiniLM
  ndcg@10: 0.6899
  recall@100: 0.5068
  map@1000: 0.4488
---
Model: DistilRoberta
  ndcg@10: 0.6310
  recall@100: 0.4795
  map@1000: 0.4074
---
Model: TinyBERT
  ndcg@10: 0.6950
  recall@100: 0.5049
  map@1000: 0.4568
---

--- Ensemble Method Performance ---
Fusing with: CombSUM (ranx method: 'sum')
Results for CombSUM:
  ndcg@10: 0.7051
  recall@100: 0.5111
  map@1000: 0.4585
---
Fusing with: CombMNZ (ranx method: 'mnz')
Results for CombMNZ:
  ndcg@10: 0.7051
  recall@100: 0.5111
  map@1000: 0.4585
---
Fusing with: RRF (ranx method: 'rrf')
Results for RRF:
  ndcg@10: 0.7109
  recall@100: 0.5133
  map@1000: 0.4658
---
Fusing with: BordaFuse (ranx method: 'bordafuse')
Results for BordaFuse:
  ndcg@10: 0.7098
  recall@100: 0.5170
  map@1000: 0.4653
---
Fusing with: LogISRFuse (ranx method: 'log_isr')
Results for LogISRFuse:
  ndcg@10: 0.6827
  recall@100: 0.5086
  map@1000: 0.4496
---


## 4. Summary of Results

In [16]:
if len(runs_to_fuse) >= 2:
    df_results = pd.DataFrame(results_summary)
    print("\n--- Results Summary Table ---")
    print(df_results.to_string())
else:
    print("No summary table generated as not enough runs were available for fusion.")


--- Results Summary Table ---
          Method   ndcg@10  recall@100  map@1000
0         MiniLM  0.689897    0.506757  0.448787
1  DistilRoberta  0.631024    0.479545  0.407447
2       TinyBERT  0.695031    0.504907  0.456773
3        CombSUM  0.705133    0.511088  0.458481
4        CombMNZ  0.705133    0.511088  0.458481
5            RRF  0.710891    0.513261  0.465823
6      BordaFuse  0.709770    0.516954  0.465329
7     LogISRFuse  0.682654    0.508634  0.449599


In [19]:
# Ensure this cell is run after the cells for Task 1 (loading runs and qrels) and Task 2 (initial ensemble evaluation)

import itertools # To get combinations of models

# --- Task 3: Analyzing Most Effective Ensemble Method ---
print("\n\n--- Task 3: Analyzing Most Effective Ensemble Method ---")

# Based on Task 2 results, RRF was among the best. Let's select it.
# If another method was better in your specific run, change these:
selected_ensemble_method_alias = "rrf" # This is the string alias for ranx.fuse
selected_ensemble_display_name = "RRF (k=60)"

# The individual runs and their names are already loaded from Task 2:
# runs_to_fuse = [run_minilm, run_distilroberta, run_tinybert]
# run_names = ["MiniLM", "DistilRoberta", "TinyBERT"]
# qrels is also loaded
# metrics = ["ndcg@10", "recall@100", "map@1000"]
# normalization_strategy = "min-max" (defined in Task 2, ensure this cell was run)

if len(runs_to_fuse) < 3: # Need at least 3 to make 3 unique pairs.
    print("WARNING: Task 3 requires at least three individual model runs for pairwise combinations. Please ensure they were loaded correctly.")
    if len(runs_to_fuse) == 2:
         print("Only two runs available, performing fusion on the single available pair.")
    else:
        print("Not enough runs to perform pairwise fusion as described (need 3 for 3 pairs).")

if len(runs_to_fuse) >= 2: # Can proceed if at least one pair can be formed
    print(f"Selected ensemble method for pairwise analysis: {selected_ensemble_display_name} (using ranx method alias: '{selected_ensemble_method_alias}')")

    pairwise_results_summary = []

    # Generate all combinations of two models
    for i, j in itertools.combinations(range(len(runs_to_fuse)), 2):
        run1 = runs_to_fuse[i]
        run2 = runs_to_fuse[j]
        name1 = run_names[i]
        name2 = run_names[j]

        current_runs_pair = [run1, run2]

        combination_name = f"{selected_ensemble_display_name} ({name1} + {name2})"
        print(f"\nFusing: {name1} and {name2} using {selected_ensemble_display_name}")

        try:
            fusion_params = {}
            if selected_ensemble_method_alias == "rrf": # Check if the selected method is RRF
                fusion_params['k'] = 60 # Set k for RRF

            fused_run_pairwise = ranx.fuse(
                runs=current_runs_pair,
                norm=normalization_strategy,
                method=selected_ensemble_method_alias, # Use the string alias here
                params=fusion_params if fusion_params else None
            )
            fused_run_pairwise.name = combination_name

            scores_pairwise = ranx.evaluate(qrels, fused_run_pairwise, metrics)

            print(f"Results for {combination_name}:")
            for metric, score in scores_pairwise.items():
                print(f"  {metric}: {score:.4f}")

            pairwise_results_summary.append({
                "Models Fused": f"{name1} + {name2}",
                "Ensemble Method": selected_ensemble_display_name,
                "ndcg@10": scores_pairwise.get("ndcg@10"),
                "recall@100": scores_pairwise.get("recall@100"),
                "map@1000": scores_pairwise.get("map@1000")
            })
        except Exception as e:
            print(f"Error applying {selected_ensemble_display_name} to {name1} + {name2}: {e}")
        print("---")

    # Display summary of pairwise fusion results
    if pairwise_results_summary:
        df_pairwise_results = pd.DataFrame(pairwise_results_summary)
        print("\n--- Pairwise Ensemble Results Summary Table ---")
        print(df_pairwise_results.to_string())
else:
    print("Skipping pairwise ensemble analysis due to insufficient loaded runs (need at least 2).")



--- Task 3: Analyzing Most Effective Ensemble Method ---
Selected ensemble method for pairwise analysis: RRF (k=60) (using ranx method alias: 'rrf')

Fusing: MiniLM and DistilRoberta using RRF (k=60)
Results for RRF (k=60) (MiniLM + DistilRoberta):
  ndcg@10: 0.6994
  recall@100: 0.5092
  map@1000: 0.4525
---

Fusing: MiniLM and TinyBERT using RRF (k=60)
Results for RRF (k=60) (MiniLM + TinyBERT):
  ndcg@10: 0.7053
  recall@100: 0.5112
  map@1000: 0.4647
---

Fusing: DistilRoberta and TinyBERT using RRF (k=60)
Results for RRF (k=60) (DistilRoberta + TinyBERT):
  ndcg@10: 0.6908
  recall@100: 0.5090
  map@1000: 0.4581
---

--- Pairwise Ensemble Results Summary Table ---
               Models Fused Ensemble Method   ndcg@10  recall@100  map@1000
0    MiniLM + DistilRoberta      RRF (k=60)  0.699357    0.509248  0.452524
1         MiniLM + TinyBERT      RRF (k=60)  0.705348    0.511207  0.464686
2  DistilRoberta + TinyBERT      RRF (k=60)  0.690819    0.508953  0.458077
